# PaliGemma 2 ile küçük deney: Türkçe tabelalar, fişler, iki çözünürlük

**Ne ölçüyoruz?**

| # | Deney | Veri | Ölçüm |
|---|---|---|---|
| 1 | Türkçe soru mu, İngilizce soru mu? | 41 Türkçe tabela/etiket fotoğrafı (Wikimedia Commons), 54 soru | Doğru cevap oranı (TR soru ile EN soru) |
| 2 | Çözünürlük: 224 mü, 448 mi? | Aynı set + CORD fişleri | Doğruluk, "büyük yazı" ile "küçük yazı" ayrı ayrı, ayrıca süre |
| 3 | Türkçe karakterler (ç ğ ı i ö ş ü) | 11 tabelada `ocr` | Karakter hata oranı (CER), harfi doğru mu yazdı yoksa noktasız/çengelsiz mi |
| 4 | Fişte toplam tutar | CORD-v2 test, 50 fiş | Toplam tutar doğru mu |

Modeller: `google/paligemma2-3b-mix-224` ve `google/paligemma2-3b-mix-448`. İkisi de 3B; tek fark görüntü çözünürlüğü (256 ve 1024 görüntü tokenı).

**Başlamadan önce:**
1. *Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU*
2. Hugging Face'te iki model sayfasını da açıp lisansı kabul et: [mix-224](https://huggingface.co/google/paligemma2-3b-mix-224), [mix-448](https://huggingface.co/google/paligemma2-3b-mix-448)
3. (İsteğe bağlı) Wikimedia hız sınırına takılmamak için `data/images/` klasörünü `tabela.zip` olarak sıkıştırıp Colab'da sol menüdeki 📁 *Dosyalar* paneline sürükle.
4. Hugging Face token'ını Colab'ın sol menüsündeki 🔑 *Secrets* bölümüne `HF_TOKEN` adıyla ekle (ya da aşağıdaki giriş kutusunu kullan).

Sonra *Çalışma zamanı → Tümünü çalıştır*. T4'te toplam ~20-30 dk sürer.

In [ ]:
!pip install -q -U transformers accelerate datasets
import torch, transformers
print(transformers.__version__, torch.__version__, torch.cuda.get_device_name(0) if torch.cuda.is_available() else "GPU YOK!")

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF_TOKEN Secrets'tan alındı")
except Exception:
    from huggingface_hub import notebook_login
    notebook_login()

## 1. Veri

Tabela seti: Türkiye'de çekilmiş, CC lisanslı 41 fotoğraf. Sorular ve doğru cevaplar elle yazıldı.
- `tip = text`: doğru cevap modelin cevabının içinde geçmeli. Büyük/küçük harf ve noktalama dikkate alınmaz.
- `tip = num`: modelin cevabındaki bir sayı doğru sayıya eşit olmalı (`14,90` = `14.90`).
- `boy`: yazı fotoğrafta **büyük** mü (uzaktan okunur) yoksa **küçük** mü (ince ayrıntı)? Çözünürlük karşılaştırmasında kullanılıyor.

In [ ]:
DATA = {
 "gorseller": {
  "c00": {
   "url": "https://thumb.wikimedia.org/wikipedia/commons/thumb/5/5a/15_Temmuz_Mahallesi_tabela.jpg/1280px-15_Temmuz_Mahallesi_tabela.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
   "sayfa": "https://commons.wikimedia.org/wiki/File:15_Temmuz_Mahallesi_tabela.jpg",
   "lisans": "CC BY-SA 4.0",
   "yazar": "Ardahanian",
   "baslik": "15 Temmuz Mahallesi tabela.jpg"
  },
  "c01": {
   "url": "https://upload.wikimedia.org/wikipedia/commons/4/49/Acil_Servis_Tabelas%C4%B1_%2C_T%C3%BCrkiye.webp?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail_unscaled",
   "sayfa": "https://commons.wikimedia.org/wiki/File:Acil_Servis_Tabelas%C4%B1_,_T%C3%BCrkiye.webp",
   "lisans": "CC BY-SA 4.0",
   "yazar": "Saglıkcıormancı4",
   "baslik": "Acil Servis Tabelası , Türkiye.webp"
  },
  "c02": {
   "url": "https://thumb.wikimedia.org/wikipedia/commons/thumb/2/21/Acil_%C3%A7%C4%B1k%C4%B1%C5%9F.JPG/1280px-Acil_%C3%A7%C4%B1k%C4%B1%C5%9F.JPG?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
   "sayfa": "https://commons.wikimedia.org/wiki/File:Acil_%C3%A7%C4%B1k%C4%B1%C5%9F.JPG",
   "lisans": "CC BY-SA 4.0",
   "yazar": "Vikiçizer",
   "baslik": "Acil çıkış.JPG"
  },
  "c03": {
   "url": "https://thumb.wikimedia.org/wikipedia/commons/thumb/7/7a/Ayd%C4%B1n_Sokak_Tabelas%C4%B1.jpg/1280px-Ayd%C4%B1n_Sokak_Tabelas%C4%B1.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
   "sayfa": "https://commons.wikimedia.org/wiki/File:Ayd%C4%B1n_Sokak_Tabelas%C4%B1.jpg",
   "lisans": "CC BY-SA 4.0",
   "yazar": "Kıvırcıklüleler",
   "baslik": "Aydın Sokak Tabelası.jpg"
  },
  "c04": {
   "url": "https://thumb.wikimedia.org/wikipedia/commons/thumb/0/00/Ayd%C4%B1n_Terminali.jpg/1280px-Ayd%C4%B1n_Terminali.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
   "sayfa": "https://commons.wikimedia.org/wiki/File:Ayd%C4%B1n_Terminali.jpg",
   "lisans": "CC BY-SA 4.0",
   "yazar": "Satirdan kahraman",
   "baslik": "Aydın Terminali.jpg"
  },
  "c05": {
   "url": "https://thumb.wikimedia.org/wikipedia/commons/thumb/b/b2/Fatih_Sultan_Mehmet_Caddesi_tabelas%C4%B1%2C_Samsun.jpg/1280px-Fatih_Sultan_Mehmet_Caddesi_tabelas%C4%B1%2C_Samsun.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
   "sayfa": "https://commons.wikimedia.org/wiki/File:Fatih_Sultan_Mehmet_Caddesi_tabelas%C4%B1,_Samsun.jpg",
   "lisans": "CC BY-SA 4.0",
   "yazar": "Vikiçizer",
   "baslik": "Fatih Sultan Mehmet Caddesi tabelası, Samsun.jpg"
  },
  "c06": {
   "url": "https://thumb.wikimedia.org/wikipedia/commons/thumb/a/a1/Gondol_iskelesi_tabelas%C4%B1.jpg/1280px-Gondol_iskelesi_tabelas%C4%B1.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
   "sayfa": "https://commons.wikimedia.org/wiki/File:Gondol_iskelesi_tabelas%C4%B1.jpg",
   "lisans": "CC BY-SA 4.0",
   "yazar": "Zemxer",
   "baslik": "Gondol iskelesi tabelası.jpg"
  },
  "c07": {
   "url": "https://thumb.wikimedia.org/wikipedia/commons/thumb/7/78/Konya_t%C3%BCrk%C3%BC_sokak_levhas%C4%B1.jpg/1280px-Konya_t%C3%BCrk%C3%BC_sokak_levhas%C4%B1.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
   "sayfa": "https://commons.wikimedia.org/wiki/File:Konya_t%C3%BCrk%C3%BC_sokak_levhas%C4%B1.jpg",
   "lisans": "CC BY-SA 4.0",
   "yazar": "Satirdan kahraman",
   "baslik": "Konya türkü sokak levhası.jpg"
  },
  "c08": {
   "url": "https://thumb.wikimedia.org/wikipedia/commons/thumb/b/b2/Kurtulu%C5%9F_Yolu_ve_%C4%B0skelesi_bilgilendirme_tabelas%C4%B1.jpg/1280px-Kurtulu%C5%9F_Yolu_ve_%C4%B0skelesi_bilgilendirme_tabelas%C4%B1.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
   "sayfa": "https://commons.wikimedia.org/wiki/File:Kurtulu%C5%9F_Yolu_ve_%C4%B0skelesi_bilgilendirme_tabelas%C4%B1.jpg",
   "lisans": "CC0",
   "yazar": "Cobija",
   "baslik": "Kurtuluş Yolu ve İskelesi bilgilendirme tabelası.jpg"
  },
  "c09": {
   "url": "https://thumb.wikimedia.org/wikipedia/commons/thumb/3/36/K%C3%B6pr%C3%BCl%C3%BC_Mehmet_Pa%C5%9Fa_Sokak_tabelas%C4%B1%2C_Samsun.jpg/1280px-K%C3%B6pr%C3%BCl%C3%BC_Mehmet_Pa%C5%9Fa_Sokak_tabelas%C4%B1%2C_Samsun.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
   "sayfa": "https://commons.wikimedia.org/wiki/File:K%C3%B6pr%C3%BCl%C3%BC_Mehmet_Pa%C5%9Fa_Sokak_tabelas%C4%B1,_Samsun.jpg",
   "lisans": "CC BY-SA 4.0",
   "yazar": "Vikiçizer",
   "baslik": "Köprülü Mehmet Paşa Sokak tabelası, Samsun.jpg"
  },
  "c10": {
   "url": "https://thumb.wikimedia.org/wikipedia/commons/thumb/1/1e/Mevlana_m%C3%BCzesi_tabela.jpg/1280px-Mevlana_m%C3%BCzesi_tabela.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
   "sayfa": "https://commons.wikimedia.org/wiki/File:Mevlana_m%C3%BCzesi_tabela.jpg",
   "lisans": "CC BY-SA 4.0",
   "yazar": "Zemxer",
   "baslik": "Mevlana müzesi tabela.jpg"
  },
  "c14": {
   "url": "https://thumb.wikimedia.org/wikipedia/commons/thumb/e/ec/SAKLI_EV_SOKA%C4%9EI_-_panoramio.jpg/1280px-SAKLI_EV_SOKA%C4%9EI_-_panoramio.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
   "sayfa": "https://commons.wikimedia.org/wiki/File:SAKLI_EV_SOKA%C4%9EI_-_panoramio.jpg",
   "lisans": "CC BY-SA 3.0",
   "yazar": "uğur baysan",
   "baslik": "SAKLI EV SOKAĞI - panoramio.jpg"
  },
  "c15": {
   "url": "https://thumb.wikimedia.org/wikipedia/commons/thumb/3/30/Sokak_levhas%C4%B1.jpg/1280px-Sokak_levhas%C4%B1.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
   "sayfa": "https://commons.wikimedia.org/wiki/File:Sokak_levhas%C4%B1.jpg",
   "lisans": "CC0",
   "yazar": "Satirdan kahraman",
   "baslik": "Sokak levhası.jpg"
  },
  "c17": {
   "url": "https://thumb.wikimedia.org/wikipedia/commons/thumb/4/4e/Vali_muammerbey_caddesi.jpg/1280px-Vali_muammerbey_caddesi.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
   "sayfa": "https://commons.wikimedia.org/wiki/File:Vali_muammerbey_caddesi.jpg",
   "lisans": "CC BY-SA 4.0",
   "yazar": "Satirdan kahraman",
   "baslik": "Vali muammerbey caddesi.jpg"
  },
  "c18": {
   "url": "https://thumb.wikimedia.org/wikipedia/commons/thumb/5/51/Zeyniler_K%C3%B6y%C3%BC%2C_Re%C5%9Fat_Nuri_G%C3%BCntekin_Soka%C4%9F%C4%B1.jpg/1280px-Zeyniler_K%C3%B6y%C3%BC%2C_Re%C5%9Fat_Nuri_G%C3%BCntekin_Soka%C4%9F%C4%B1.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
   "sayfa": "https://commons.wikimedia.org/wiki/File:Zeyniler_K%C3%B6y%C3%BC,_Re%C5%9Fat_Nuri_G%C3%BCntekin_Soka%C4%9F%C4%B1.jpg",
   "lisans": "CC BY-SA 4.0",
   "yazar": "Basak",
   "baslik": "Zeyniler Köyü, Reşat Nuri Güntekin Sokağı.jpg"
  },
  "c19": {
   "url": "https://thumb.wikimedia.org/wikipedia/commons/thumb/2/22/%C3%87%C4%B1narc%C4%B1k_Sign_03.06.2025_01.jpg/1280px-%C3%87%C4%B1narc%C4%B1k_Sign_03.06.2025_01.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
   "sayfa": "https://commons.wikimedia.org/wiki/File:%C3%87%C4%B1narc%C4%B1k_Sign_03.06.2025_01.jpg",
   "lisans": "CC BY 4.0",
   "yazar": "Wiki leylek",
   "baslik": "Çınarcık Sign 03.06.2025 01.jpg"
  },
  "c20": {
   "url": "https://thumb.wikimedia.org/wikipedia/commons/thumb/3/3d/AK%C3%87AY_MAK%C4%B0NA.jpg/1280px-AK%C3%87AY_MAK%C4%B0NA.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
   "sayfa": "https://commons.wikimedia.org/wiki/File:AK%C3%87AY_MAK%C4%B0NA.jpg",
   "lisans": "CC BY-SA 4.0",
   "yazar": "Sami zb",
   "baslik": "AKÇAY MAKİNA.jpg"
  },
  "c21": {
   "url": "https://thumb.wikimedia.org/wikipedia/commons/thumb/7/7c/Arsac%C4%B1_balaban_emlak_-_panoramio.jpg/1280px-Arsac%C4%B1_balaban_emlak_-_panoramio.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
   "sayfa": "https://commons.wikimedia.org/wiki/File:Arsac%C4%B1_balaban_emlak_-_panoramio.jpg",
   "lisans": "CC BY-SA 3.0",
   "yazar": "balabanemlak",
   "baslik": "Arsacı balaban emlak - panoramio.jpg"
  },
  "c23": {
   "url": "https://upload.wikimedia.org/wikipedia/commons/c/cd/Bayat_ekmek_ask%C4%B1s%C4%B1.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail_unscaled",
   "sayfa": "https://commons.wikimedia.org/wiki/File:Bayat_ekmek_ask%C4%B1s%C4%B1.jpg",
   "lisans": "CC BY-SA 4.0",
   "yazar": "Sadrettin",
   "baslik": "Bayat ekmek askısı.jpg"
  },
  "c24": {
   "url": "https://thumb.wikimedia.org/wikipedia/commons/thumb/4/47/Dikkat_Hareket_Eden_Rayl%C4%B1_Y%C3%BCr%C3%BCme_Yolu.jpg/1280px-Dikkat_Hareket_Eden_Rayl%C4%B1_Y%C3%BCr%C3%BCme_Yolu.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
   "sayfa": "https://commons.wikimedia.org/wiki/File:Dikkat_Hareket_Eden_Rayl%C4%B1_Y%C3%BCr%C3%BCme_Yolu.jpg",
   "lisans": "CC0",
   "yazar": "Satirdan kahraman",
   "baslik": "Dikkat Hareket Eden Raylı Yürüme Yolu.jpg"
  },
  "c25": {
   "url": "https://thumb.wikimedia.org/wikipedia/commons/thumb/3/38/Directional_sign_in_K%C3%BClt%C3%BCrpark.jpg/1280px-Directional_sign_in_K%C3%BClt%C3%BCrpark.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
   "sayfa": "https://commons.wikimedia.org/wiki/File:Directional_sign_in_K%C3%BClt%C3%BCrpark.jpg",
   "lisans": "CC BY-SA 4.0",
   "yazar": "BSRF",
   "baslik": "Directional sign in Kültürpark.jpg"
  },
  "c26": {
   "url": "https://thumb.wikimedia.org/wikipedia/commons/thumb/e/e8/Don%27t_Smoke_written_in_Turkish_in_Marma_Otel_in_Maltepe_University%2C_Istanbul.JPG/1280px-Don%27t_Smoke_written_in_Turkish_in_Marma_Otel_in_Maltepe_University%2C_Istanbul.JPG?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
   "sayfa": "https://commons.wikimedia.org/wiki/File:Don%27t_Smoke_written_in_Turkish_in_Marma_Otel_in_Maltepe_University,_Istanbul.JPG",
   "lisans": "CC BY-SA 3.0",
   "yazar": "Ibrahim Husain Meraj",
   "baslik": "Don't Smoke written in Turkish in Marma Otel in Maltepe University, Istanbul.JPG"
  },
  "c28": {
   "url": "https://upload.wikimedia.org/wikipedia/commons/2/21/D%C3%9CKKAN.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail_unscaled",
   "sayfa": "https://commons.wikimedia.org/wiki/File:D%C3%9CKKAN.jpg",
   "lisans": "CC BY-SA 4.0",
   "yazar": "Dağhan tabela",
   "baslik": "DÜKKAN.jpg"
  },
  "c29": {
   "url": "https://thumb.wikimedia.org/wikipedia/commons/thumb/6/69/Faz%C4%B1l_Kad%C4%B1_Gen%C3%A7lik_Merkezi.jpg/1280px-Faz%C4%B1l_Kad%C4%B1_Gen%C3%A7lik_Merkezi.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
   "sayfa": "https://commons.wikimedia.org/wiki/File:Faz%C4%B1l_Kad%C4%B1_Gen%C3%A7lik_Merkezi.jpg",
   "lisans": "CC BY-SA 4.0",
   "yazar": "Vikiçizer",
   "baslik": "Fazıl Kadı Gençlik Merkezi.jpg"
  },
  "c30": {
   "url": "https://thumb.wikimedia.org/wikipedia/commons/thumb/1/1e/GalataKulesiTabelas%C4%B1.jpg/1280px-GalataKulesiTabelas%C4%B1.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
   "sayfa": "https://commons.wikimedia.org/wiki/File:GalataKulesiTabelas%C4%B1.jpg",
   "lisans": "Public domain",
   "yazar": "Vikiçizer at Turkish Wikipedia",
   "baslik": "GalataKulesiTabelası.jpg"
  },
  "c31": {
   "url": "https://thumb.wikimedia.org/wikipedia/commons/thumb/b/b5/Hamsi.jpg/1280px-Hamsi.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
   "sayfa": "https://commons.wikimedia.org/wiki/File:Hamsi.jpg",
   "lisans": "CC BY-SA 3.0",
   "yazar": "Original uploader was Citrat at tr.wikipedia",
   "baslik": "Hamsi.jpg"
  },
  "c32": {
   "url": "https://upload.wikimedia.org/wikipedia/commons/0/07/KKTC_Yol_Tabelasi_1.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail_unscaled",
   "sayfa": "https://commons.wikimedia.org/wiki/File:KKTC_Yol_Tabelasi_1.jpg",
   "lisans": "Public domain",
   "yazar": "Seksen iki yüz kırk beş",
   "baslik": "KKTC Yol Tabelasi 1.jpg"
  },
  "c33": {
   "url": "https://upload.wikimedia.org/wikipedia/commons/c/c7/KOBRA_B%C4%B0LG%C4%B0SAYAR_-_panoramio.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail_unscaled",
   "sayfa": "https://commons.wikimedia.org/wiki/File:KOBRA_B%C4%B0LG%C4%B0SAYAR_-_panoramio.jpg",
   "lisans": "CC BY 3.0",
   "yazar": "sonmez81",
   "baslik": "KOBRA BİLGİSAYAR - panoramio.jpg"
  },
  "c36": {
   "url": "https://thumb.wikimedia.org/wikipedia/commons/thumb/d/d8/Palamut_ithal.jpg/1280px-Palamut_ithal.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
   "sayfa": "https://commons.wikimedia.org/wiki/File:Palamut_ithal.jpg",
   "lisans": "CC BY-SA 3.0",
   "yazar": "Citrat at Turkish Wikipedia",
   "baslik": "Palamut ithal.jpg"
  },
  "c38": {
   "url": "https://thumb.wikimedia.org/wikipedia/commons/thumb/0/0e/Yerli_Somon.jpg/1280px-Yerli_Somon.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
   "sayfa": "https://commons.wikimedia.org/wiki/File:Yerli_Somon.jpg",
   "lisans": "CC BY-SA 3.0",
   "yazar": "Citrat at Turkish Wikipedia",
   "baslik": "Yerli Somon.jpg"
  },
  "c40": {
   "url": "https://thumb.wikimedia.org/wikipedia/commons/thumb/1/1e/Alabalik1.jpg/1280px-Alabalik1.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
   "sayfa": "https://commons.wikimedia.org/wiki/File:Alabalik1.jpg",
   "lisans": "CC BY-SA 3.0",
   "yazar": "Citrat at Turkish Wikipedia",
   "baslik": "Alabalik1.jpg"
  },
  "c41": {
   "url": "https://thumb.wikimedia.org/wikipedia/commons/thumb/6/6f/Beypazar%C4%B1_kurusu.jpg/1280px-Beypazar%C4%B1_kurusu.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
   "sayfa": "https://commons.wikimedia.org/wiki/File:Beypazar%C4%B1_kurusu.jpg",
   "lisans": "CC BY-SA 4.0",
   "yazar": "E4024",
   "baslik": "Beypazarı kurusu.jpg"
  },
  "c42": {
   "url": "https://thumb.wikimedia.org/wikipedia/commons/thumb/0/01/Cafe_markiz.JPG/1280px-Cafe_markiz.JPG?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
   "sayfa": "https://commons.wikimedia.org/wiki/File:Cafe_markiz.JPG",
   "lisans": "Public domain",
   "yazar": "Homonihilis",
   "baslik": "Cafe markiz.JPG"
  },
  "c43": {
   "url": "https://thumb.wikimedia.org/wikipedia/commons/thumb/6/6a/Caferzade.jpg/1280px-Caferzade.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
   "sayfa": "https://commons.wikimedia.org/wiki/File:Caferzade.jpg",
   "lisans": "CC BY-SA 4.0",
   "yazar": "E4024",
   "baslik": "Caferzade.jpg"
  },
  "c44": {
   "url": "https://thumb.wikimedia.org/wikipedia/commons/thumb/8/89/Distances_from_Milion_Marker%2C_Istanbul.JPG/1280px-Distances_from_Milion_Marker%2C_Istanbul.JPG?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
   "sayfa": "https://commons.wikimedia.org/wiki/File:Distances_from_Milion_Marker,_Istanbul.JPG",
   "lisans": "CC BY-SA 4.0",
   "yazar": "Cpandmg",
   "baslik": "Distances from Milion Marker, Istanbul.JPG"
  },
  "c45": {
   "url": "https://upload.wikimedia.org/wikipedia/commons/1/13/Entranceforbidden.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail_unscaled",
   "sayfa": "https://commons.wikimedia.org/wiki/File:Entranceforbidden.jpg",
   "lisans": "Public domain",
   "yazar": "Spiderone",
   "baslik": "Entranceforbidden.jpg"
  },
  "c46": {
   "url": "https://thumb.wikimedia.org/wikipedia/commons/thumb/4/41/Food_market_in_%C3%87ankaya.jpg/1280px-Food_market_in_%C3%87ankaya.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
   "sayfa": "https://commons.wikimedia.org/wiki/File:Food_market_in_%C3%87ankaya.jpg",
   "lisans": "CC BY-SA 4.0",
   "yazar": "E4024",
   "baslik": "Food market in Çankaya.jpg"
  },
  "c47": {
   "url": "https://thumb.wikimedia.org/wikipedia/commons/thumb/0/06/Istanbul_Cagaloglu_hamami_entrance.jpg/1280px-Istanbul_Cagaloglu_hamami_entrance.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
   "sayfa": "https://commons.wikimedia.org/wiki/File:Istanbul_Cagaloglu_hamami_entrance.jpg",
   "lisans": "CC BY-SA 3.0",
   "yazar": "Rose28357",
   "baslik": "Istanbul Cagaloglu hamami entrance.jpg"
  },
  "c48": {
   "url": "https://thumb.wikimedia.org/wikipedia/commons/thumb/9/93/Istanbul_City_Sign_in_Kad%C4%B1k%C3%B6y_21.08.2025_01.jpg/1280px-Istanbul_City_Sign_in_Kad%C4%B1k%C3%B6y_21.08.2025_01.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
   "sayfa": "https://commons.wikimedia.org/wiki/File:Istanbul_City_Sign_in_Kad%C4%B1k%C3%B6y_21.08.2025_01.jpg",
   "lisans": "CC BY 4.0",
   "yazar": "Wiki leylek",
   "baslik": "Istanbul City Sign in Kadıköy 21.08.2025 01.jpg"
  },
  "c49": {
   "url": "https://thumb.wikimedia.org/wikipedia/commons/thumb/a/a1/Kuru_yemisci.jpg/1280px-Kuru_yemisci.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
   "sayfa": "https://commons.wikimedia.org/wiki/File:Kuru_yemisci.jpg",
   "lisans": "CC BY-SA 4.0",
   "yazar": "E4024",
   "baslik": "Kuru yemisci.jpg"
  },
  "c50": {
   "url": "https://thumb.wikimedia.org/wikipedia/commons/thumb/7/74/Kuruyemisci.jpg/1280px-Kuruyemisci.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
   "sayfa": "https://commons.wikimedia.org/wiki/File:Kuruyemisci.jpg",
   "lisans": "CC BY-SA 4.0",
   "yazar": "E4024",
   "baslik": "Kuruyemisci.jpg"
  }
 },
 "sorular": [
  {
   "qid": "q00",
   "img": "c01",
   "tr": "Tabelada ne yazıyor?",
   "en": "What does the sign say?",
   "gold": [
    "acil servis"
   ],
   "tip": "text",
   "boy": "büyük"
  },
  {
   "qid": "q01",
   "img": "c02",
   "tr": "Tabelada ne yazıyor?",
   "en": "What does the sign say?",
   "gold": [
    "acil çıkış"
   ],
   "tip": "text",
   "boy": "büyük"
  },
  {
   "qid": "q02",
   "img": "c03",
   "tr": "Sokağın numarası kaç?",
   "en": "What is the street number?",
   "gold": [
    2151
   ],
   "tip": "num",
   "boy": "büyük"
  },
  {
   "qid": "q03",
   "img": "c03",
   "tr": "Hangi mahallede?",
   "en": "Which neighborhood is this?",
   "gold": [
    "camiatik"
   ],
   "tip": "text",
   "boy": "küçük"
  },
  {
   "qid": "q04",
   "img": "c04",
   "tr": "Bu terminal hangi şehirde?",
   "en": "Which city is this terminal in?",
   "gold": [
    "aydın"
   ],
   "tip": "text",
   "boy": "büyük"
  },
  {
   "qid": "q05",
   "img": "c05",
   "tr": "Caddenin adı ne?",
   "en": "What is the name of the street?",
   "gold": [
    "fatih sultan"
   ],
   "tip": "text",
   "boy": "büyük"
  },
  {
   "qid": "q06",
   "img": "c05",
   "tr": "Hangi mahallede?",
   "en": "Which neighborhood is this?",
   "gold": [
    "pazar"
   ],
   "tip": "text",
   "boy": "küçük"
  },
  {
   "qid": "q07",
   "img": "c07",
   "tr": "Sokağın adı ne?",
   "en": "What is the name of the street?",
   "gold": [
    "türkü"
   ],
   "tip": "text",
   "boy": "büyük"
  },
  {
   "qid": "q08",
   "img": "c09",
   "tr": "Sokağın adı ne?",
   "en": "What is the name of the street?",
   "gold": [
    "köprülü mehmet paşa"
   ],
   "tip": "text",
   "boy": "büyük"
  },
  {
   "qid": "q09",
   "img": "c09",
   "tr": "Hangi mahallede?",
   "en": "Which neighborhood is this?",
   "gold": [
    "kökçüoğlu"
   ],
   "tip": "text",
   "boy": "küçük"
  },
  {
   "qid": "q10",
   "img": "c10",
   "tr": "Düz gidersem nereye varırım?",
   "en": "Where does the straight arrow lead?",
   "gold": [
    "şehir merkezi"
   ],
   "tip": "text",
   "boy": "küçük"
  },
  {
   "qid": "q11",
   "img": "c15",
   "tr": "Caddenin adı ne?",
   "en": "What is the name of the avenue?",
   "gold": [
    "kazım karabekir"
   ],
   "tip": "text",
   "boy": "büyük"
  },
  {
   "qid": "q12",
   "img": "c17",
   "tr": "Alttaki tabelada ne yazıyor?",
   "en": "What does the lower sign say?",
   "gold": [
    "park etmek yasak"
   ],
   "tip": "text",
   "boy": "büyük"
  },
  {
   "qid": "q13",
   "img": "c17",
   "tr": "Caddenin adı ne?",
   "en": "What is the name of the avenue?",
   "gold": [
    "vali muammerbey"
   ],
   "tip": "text",
   "boy": "büyük"
  },
  {
   "qid": "q14",
   "img": "c18",
   "tr": "Sokağın adı ne?",
   "en": "What is the name of the street?",
   "gold": [
    "reşat nuri güntekin"
   ],
   "tip": "text",
   "boy": "büyük"
  },
  {
   "qid": "q15",
   "img": "c19",
   "tr": "Büyük harflerle hangi yerin adı yazıyor?",
   "en": "Which place name is written in big letters?",
   "gold": [
    "çınarcık"
   ],
   "tip": "text",
   "boy": "büyük"
  },
  {
   "qid": "q16",
   "img": "c19",
   "tr": "Büyük yazının altında ne yazıyor?",
   "en": "What is written below the big letters?",
   "gold": [
    "marmaranın incisi"
   ],
   "tip": "text",
   "boy": "küçük"
  },
  {
   "qid": "q17",
   "img": "c20",
   "tr": "Dükkanın adı ne?",
   "en": "What is the name of the shop?",
   "gold": [
    "akçay makine"
   ],
   "tip": "text",
   "boy": "büyük"
  },
  {
   "qid": "q18",
   "img": "c20",
   "tr": "Telefon numarası ne?",
   "en": "What is the phone number?",
   "gold": [
    "546 33 36"
   ],
   "tip": "text",
   "boy": "büyük"
  },
  {
   "qid": "q19",
   "img": "c21",
   "tr": "Işıklı tabelada ne yazıyor?",
   "en": "What does the illuminated sign say?",
   "gold": [
    "balaban emlak"
   ],
   "tip": "text",
   "boy": "büyük"
  },
  {
   "qid": "q20",
   "img": "c21",
   "tr": "Sigorta şirketinin adı ne?",
   "en": "What is the name of the insurance company?",
   "gold": [
    "anadolu sigorta"
   ],
   "tip": "text",
   "boy": "küçük"
  },
  {
   "qid": "q21",
   "img": "c23",
   "tr": "Tabelada ne yazıyor?",
   "en": "What does the sign say?",
   "gold": [
    "bayat ekmek askısı"
   ],
   "tip": "text",
   "boy": "büyük"
  },
  {
   "qid": "q22",
   "img": "c24",
   "tr": "Tabelada ne yazıyor?",
   "en": "What does the sign say?",
   "gold": [
    "dikkat hareket eden raylı yürüme yolu"
   ],
   "tip": "text",
   "boy": "büyük"
  },
  {
   "qid": "q23",
   "img": "c25",
   "tr": "En alttaki okta en son hangi yer yazıyor?",
   "en": "What is the last place written on the bottom arrow?",
   "gold": [
    "26 ağustos kapısı"
   ],
   "tip": "text",
   "boy": "küçük"
  },
  {
   "qid": "q24",
   "img": "c26",
   "tr": "Burada ne yasak?",
   "en": "What is forbidden here?",
   "gold": [
    "sigara"
   ],
   "tip": "text",
   "boy": "büyük"
  },
  {
   "qid": "q25",
   "img": "c26",
   "tr": "Kaç sayılı kanundan bahsediliyor?",
   "en": "Which law number is mentioned?",
   "gold": [
    4207
   ],
   "tip": "num",
   "boy": "küçük"
  },
  {
   "qid": "q26",
   "img": "c28",
   "tr": "Firmanın adı ne?",
   "en": "What is the name of the company?",
   "gold": [
    "dağhan"
   ],
   "tip": "text",
   "boy": "büyük"
  },
  {
   "qid": "q27",
   "img": "c29",
   "tr": "Binanın adı ne?",
   "en": "What is the name of the building?",
   "gold": [
    "fazıl kadı"
   ],
   "tip": "text",
   "boy": "küçük"
  },
  {
   "qid": "q28",
   "img": "c30",
   "tr": "Kitabede hangi tarih yazıyor?",
   "en": "What date is written on the plaque?",
   "gold": [
    "29 mayıs 1453"
   ],
   "tip": "text",
   "boy": "büyük"
  },
  {
   "qid": "q29",
   "img": "c31",
   "tr": "Hamsi kaç lira?",
   "en": "How much is the hamsi?",
   "gold": [
    4
   ],
   "tip": "num",
   "boy": "büyük"
  },
  {
   "qid": "q30",
   "img": "c32",
   "tr": "G. Mağusa kaç km?",
   "en": "How many km to G. Mağusa?",
   "gold": [
    55
   ],
   "tip": "num",
   "boy": "küçük"
  },
  {
   "qid": "q31",
   "img": "c32",
   "tr": "Tabeladaki ilk yer neresi?",
   "en": "What is the first place listed on the sign?",
   "gold": [
    "haspolat"
   ],
   "tip": "text",
   "boy": "küçük"
  },
  {
   "qid": "q32",
   "img": "c36",
   "tr": "Palamut kaç lira?",
   "en": "How much is the palamut?",
   "gold": [
    5
   ],
   "tip": "num",
   "boy": "büyük"
  },
  {
   "qid": "q33",
   "img": "c38",
   "tr": "Somon kaç lira?",
   "en": "How much is the somon?",
   "gold": [
    8
   ],
   "tip": "num",
   "boy": "büyük"
  },
  {
   "qid": "q34",
   "img": "c40",
   "tr": "Etikette hangi balığın adı yazıyor?",
   "en": "Which fish name is written on the label?",
   "gold": [
    "alabalık"
   ],
   "tip": "text",
   "boy": "büyük"
  },
  {
   "qid": "q35",
   "img": "c40",
   "tr": "Etiketteki fiyat ne?",
   "en": "What is the price on the label?",
   "gold": [
    7
   ],
   "tip": "num",
   "boy": "büyük"
  },
  {
   "qid": "q36",
   "img": "c41",
   "tr": "Beypazarı kurusunun fiyatı ne?",
   "en": "What is the price of the Beypazarı kurusu?",
   "gold": [
    14.9
   ],
   "tip": "num",
   "boy": "büyük"
  },
  {
   "qid": "q37",
   "img": "c43",
   "tr": "Tulumba kaç lira?",
   "en": "How much is the tulumba?",
   "gold": [
    12
   ],
   "tip": "num",
   "boy": "küçük"
  },
  {
   "qid": "q38",
   "img": "c44",
   "tr": "Londra kaç km uzaklıkta?",
   "en": "How far is London in km?",
   "gold": [
    2502
   ],
   "tip": "num",
   "boy": "küçük"
  },
  {
   "qid": "q39",
   "img": "c44",
   "tr": "Moskova kaç km uzaklıkta?",
   "en": "How far is Moscow in km?",
   "gold": [
    1757
   ],
   "tip": "num",
   "boy": "küçük"
  },
  {
   "qid": "q40",
   "img": "c45",
   "tr": "Tabelada Türkçe ne yazıyor?",
   "en": "What does the Turkish text on the sign say?",
   "gold": [
    "askeri güvenlik bölgesi"
   ],
   "tip": "text",
   "boy": "büyük"
  },
  {
   "qid": "q41",
   "img": "c47",
   "tr": "Hamamın adı ne?",
   "en": "What is the name of the hamam?",
   "gold": [
    "cağaloğlu"
   ],
   "tip": "text",
   "boy": "küçük"
  },
  {
   "qid": "q42",
   "img": "c47",
   "tr": "Hamam hangi yıl kurulmuş?",
   "en": "In which year was the hamam founded?",
   "gold": [
    1741
   ],
   "tip": "num",
   "boy": "küçük"
  },
  {
   "qid": "q43",
   "img": "c48",
   "tr": "Hangi şehrin adı yazıyor?",
   "en": "Which city name is written?",
   "gold": [
    "istanbul"
   ],
   "tip": "text",
   "boy": "büyük"
  },
  {
   "qid": "q44",
   "img": "c49",
   "tr": "Soslu mısır kaç lira?",
   "en": "How much is the soslu mısır?",
   "gold": [
    14
   ],
   "tip": "num",
   "boy": "büyük"
  },
  {
   "qid": "q45",
   "img": "c49",
   "tr": "Elit karışık kaç lira?",
   "en": "How much is the elit karışık?",
   "gold": [
    25
   ],
   "tip": "num",
   "boy": "büyük"
  },
  {
   "qid": "q46",
   "img": "c50",
   "tr": "Yaprak kayısı kaç lira?",
   "en": "How much is the yaprak kayısı?",
   "gold": [
    27.9
   ],
   "tip": "num",
   "boy": "küçük"
  },
  {
   "qid": "q47",
   "img": "c00",
   "tr": "Tabelada hangi ilçe yazıyor?",
   "en": "Which district is written on the sign?",
   "gold": [
    "esenler"
   ],
   "tip": "text",
   "boy": "küçük"
  },
  {
   "qid": "q48",
   "img": "c00",
   "tr": "Askı ilanı başlangıç tarihi ne?",
   "en": "What is the start date of the notice?",
   "gold": [
    "26.08.2020"
   ],
   "tip": "text",
   "boy": "küçük"
  },
  {
   "qid": "q49",
   "img": "c06",
   "tr": "Yön tabelasında ne yazıyor?",
   "en": "What does the direction sign say?",
   "gold": [
    "gondol"
   ],
   "tip": "text",
   "boy": "küçük"
  },
  {
   "qid": "q50",
   "img": "c08",
   "tr": "Panonun başlığı ne?",
   "en": "What is the title of the board?",
   "gold": [
    "kurtuluş yolu"
   ],
   "tip": "text",
   "boy": "küçük"
  },
  {
   "qid": "q51",
   "img": "c14",
   "tr": "Restoranın adı ne?",
   "en": "What is the name of the restaurant?",
   "gold": [
    "saklı ev"
   ],
   "tip": "text",
   "boy": "küçük"
  },
  {
   "qid": "q52",
   "img": "c42",
   "tr": "Kafenin adı ne?",
   "en": "What is the name of the cafe?",
   "gold": [
    "robert's coffee",
    "roberts coffee"
   ],
   "tip": "text",
   "boy": "büyük"
  },
  {
   "qid": "q53",
   "img": "c33",
   "tr": "Mavi tabelada ne yazıyor?",
   "en": "What does the blue sign say?",
   "gold": [
    "adsl"
   ],
   "tip": "text",
   "boy": "küçük"
  }
 ],
 "ocr": {
  "c01": "ACİL SERVİS Emergency",
  "c02": "ACİL ÇIKIŞ",
  "c03": "T.C. AYDIN BÜYÜKŞEHİR BELEDİYESİ CAMİATİK MAHALLESİ 2151 SOKAK (TN: 43)",
  "c09": "KÖPRÜLÜ MEHMET PAŞA SOKAK Kökçüoğlu Mahallesi (TN:11)",
  "c18": "REŞAT NURİ GÜNTEKİN SOKAK",
  "c19": "ÇINARCIK MARMARANIN İNCİSİ",
  "c20": "AKÇAY MAKİNE ADLER PFAFF HIGHLEAD SEIKO TYPICAL brother MAKİNE ALIM SATIM YEDEK PARÇA İMALAT ve TAMİR Telfax:546 33 36",
  "c23": "BAYAT EKMEK ASKISI",
  "c24": "DİKKAT HAREKET EDEN RAYLI YÜRÜME YOLU",
  "c26": "SİGARA İÇİLMEZ Burada tütün ürünleri tüketilmesi yasaktır. Tüketenlere ve tüketilmesine göz yuman sorumlulara 4207 sayılı Kanun uyarınca idari para cezası uygulanır.",
  "c30": "29 MAYIS 1453 SALI SABAHI CENEVİZLERİN GALATA KOLONİSİ ANAHTARLARINI FATİH SULTAN MEHMED'E TAKDİM ETMİŞ VE GALATANIN TESLİMİ 1 HAZİRAN CUMA GÜNÜ TAMAMLANMIŞDIR. İSTANBUL FETHİ DERNEĞİ 1953"
 },
 "altyazi": [
  "c06",
  "c10",
  "c23",
  "c31",
  "c41",
  "c46"
 ]
}
print(len(DATA["gorseller"]), "görsel,", len(DATA["sorular"]), "soru,", len(DATA["ocr"]), "OCR örneği")

In [ ]:
# Görseller sırayla şuralardan alınır:
#   1) /content/tabela.zip (sol menü 📁 Dosyalar'a sürükle-bırak; varsa)
#   2) GitHub reposu (GITHUB_RAW tanımlıysa)
#   3) Wikimedia Commons (429 "çok fazla istek" gelirse bekleyip tekrar dener)
GITHUB_RAW = None  # repo açılınca: "https://raw.githubusercontent.com/<kullanıcı>/<repo>/main/papers/01-paligemma2/experiment/data/images"
import requests, io, time, zipfile
from PIL import Image
os.makedirs("/content/tabela", exist_ok=True)
if os.path.exists("/content/tabela.zip"):
    zipfile.ZipFile("/content/tabela.zip").extractall("/content/tabela")
    print("tabela.zip açıldı")

UA = {"User-Agent": "yz50-paligemma2-deney/0.1 (egitim amacli)"}
def indir(url, p, deneme=6):
    for k in range(deneme):
        r = requests.get(url, headers=UA, timeout=60)
        if r.status_code == 429:
            bekle = int(r.headers.get("Retry-After", 0) or 0) or 10 * (k + 1)
            print(f"  429, {bekle} sn bekleniyor..."); time.sleep(bekle); continue
        r.raise_for_status(); open(p, "wb").write(r.content); return
    raise RuntimeError("Wikimedia izin vermiyor: tabela.zip'i yükleyip bu hücreyi tekrar çalıştır.")

IMG = {}
for gid, g in DATA["gorseller"].items():
    p = f"/content/tabela/{gid}.jpg"
    if not os.path.exists(p) and GITHUB_RAW:
        r = requests.get(f"{GITHUB_RAW}/{gid}.jpg", timeout=60)
        if r.ok: open(p, "wb").write(r.content)
    if not os.path.exists(p):
        indir(g["url"], p); time.sleep(1.5)
    IMG[gid] = Image.open(p).convert("RGB")
print(len(IMG), "görsel hazır")

In [ ]:
# Fişler: CORD-v2 test (CC BY 4.0). Endonezya fişleri, yazı İngilizce/Endonezce.
import json as _json
from datasets import load_dataset
N_FIS = 50
cord = []
for ex in load_dataset("naver-clova-ix/cord-v2", split="test", streaming=True):
    tot = _json.loads(ex["ground_truth"])["gt_parse"].get("total", {})
    tp = tot.get("total_price") if isinstance(tot, dict) else None
    if isinstance(tp, str) and any(ch.isdigit() for ch in tp):
        cord.append(dict(fid=f"f{len(cord):02d}", img=ex["image"].convert("RGB"), gold=tp))
    if len(cord) == N_FIS: break
print(len(cord), "fiş; örnek toplam:", cord[0]["gold"], "| boyut:", cord[0]["img"].size)

## 2. Puanlama

Türkçe için iki ayrı puan tutuyoruz:
- **katı**: `ç ğ ı i ö ş ü` harfleri birebir doğru olmalı (`türkü` ≠ `turku`)
- **esnek**: harfler katlanıyor (`ş→s`, `ı→i`, ...). Model metni okumuş ama Türkçe harfleri kaçırmışsa esnekte doğru, katıda yanlış sayılır.

Aradaki fark doğrudan "Türkçe harf kaybı"nı gösteriyor.

In [ ]:
import re, difflib

TR_FOLD = str.maketrans("çğıöşüiâîû", "cgiosuiaiu")
TR_PAIR = {"ç":"c","ğ":"g","ı":"i","ö":"o","ş":"s","ü":"u","i":"ı"}   # i -> ı: 'İ' yerine 'I' yazılması
def tr_lower(s):
    return s.replace("I", "ı").replace("İ", "i").lower()

def norm(s, fold=False):
    s = tr_lower(str(s))
    if fold: s = s.translate(TR_FOLD)
    return re.sub(r"[^0-9a-zçğıöşüâîû]", "", s)

def num_eq(tok, g):
    tok = tok.strip(".,")
    adaylar = set()
    rakam = re.sub(r"[.,]", "", tok)
    if rakam: adaylar.add(float(rakam))
    m = re.fullmatch(r"(\d+)[.,](\d{1,2})", tok)
    if m: adaylar.add(float(m.group(1) + "." + m.group(2)))
    return any(abs(a - g) < 1e-6 for a in adaylar)

def puanla(pred, golds, tip):
    """(katı, esnek) -> 0/1"""
    if tip == "num":
        ok = any(num_eq(t, g) for t in re.findall(r"\d[\d.,]*", pred) for g in golds)
        return int(ok), int(ok)
    kati  = any(norm(g) in norm(pred) for g in golds)
    esnek = any(norm(g, True) in norm(pred, True) for g in golds)
    return int(kati), int(esnek)

def fis_dogru(pred, gold):
    g = re.sub(r"\D", "", gold)
    return int(any(re.sub(r"\D", "", t) == g for t in re.findall(r"\d[\d.,]*", pred)))

def lev(a, b):
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(prev[j] + 1, cur[j-1] + 1, prev[j-1] + (ca != cb)))
        prev = cur
    return prev[-1]

def cer(pred, gold, fold=False):
    p, g = tr_lower(" ".join(pred.split())), tr_lower(" ".join(gold.split()))
    if fold: p, g = p.translate(TR_FOLD), g.translate(TR_FOLD)
    return lev(p, g) / max(1, len(g))

def tr_harf(pred, gold):
    """Altın metindeki her Türkçe harf için: doğru / katlanmış (ş->s gibi) / başka hata"""
    p, g = tr_lower(" ".join(pred.split())), tr_lower(" ".join(gold.split()))
    dogru = katlanmis = diger = 0
    for tag, i1, i2, j1, j2 in difflib.SequenceMatcher(None, g, p, autojunk=False).get_opcodes():
        for k in range(i1, i2):
            ch = g[k]
            if ch not in TR_PAIR: continue
            if tag == "equal": dogru += 1
            elif tag == "replace" and j1 + (k - i1) < j2 and p[j1 + (k - i1)] == TR_PAIR[ch]: katlanmis += 1
            else: diger += 1
    return dogru, katlanmis, diger

In [ ]:
# Puanlama testleri (model olmadan)
assert puanla("ACIL SERVIS", ["acil servis"], "text") == (0, 1)
assert puanla("ACİL SERVİS", ["acil servis"], "text") == (1, 1)
assert puanla("14,90 TL", [14.90], "num") == (1, 1)
assert puanla("2,502 km", [2502], "num") == (1, 1)
assert puanla("19,90", [14.90], "num") == (0, 0)
assert fis_dogru("Rp 60.000", "60.000") and not fis_dogru("100,000", "28,000")
print(tr_harf("ACIL CIKIS", "ACİL ÇIKIŞ"), "-> (doğru, katlanmış, diğer)")

## 3. Model çalıştırma

`DTYPE`: varsayılan `bfloat16`. Sağlık kontrolünde çıktı boş ya da anlamsız çıkarsa `torch.float32` yap (3B model T4'e sığar ama yavaşlar). `float16` önerilmez, çünkü Gemma 2 bu tipte taşma yapabiliyor.

In [ ]:
import gc, pandas as pd
from transformers import PaliGemmaProcessor, PaliGemmaForConditionalGeneration

DTYPE = torch.bfloat16
MODELLER = ["google/paligemma2-3b-mix-224", "google/paligemma2-3b-mix-448"]

def yukle(ad):
    proc = PaliGemmaProcessor.from_pretrained(ad)
    try:
        m = PaliGemmaForConditionalGeneration.from_pretrained(ad, dtype=DTYPE, device_map="cuda")
    except TypeError:
        m = PaliGemmaForConditionalGeneration.from_pretrained(ad, torch_dtype=DTYPE, device_map="cuda")
    return proc, m.eval()

@torch.inference_mode()
def sor(proc, model, img, prompt, max_new=32):
    inp = proc(text="<image>" + prompt, images=img, return_tensors="pt").to(model.device)
    inp["pixel_values"] = inp["pixel_values"].to(DTYPE)
    n = inp["input_ids"].shape[-1]
    torch.cuda.synchronize(); t = time.time()
    out = model.generate(**inp, max_new_tokens=max_new, do_sample=False)
    torch.cuda.synchronize()
    return proc.decode(out[0][n:], skip_special_tokens=True).strip(), time.time() - t

In [ ]:
# Sağlık kontrolü: küçük modelle tek görsel
proc, model = yukle(MODELLER[0]); model._ad = MODELLER[0]
print(sor(proc, model, IMG["c02"], "ocr", 20))
print(sor(proc, model, IMG["c02"], "answer tr Tabelada ne yazıyor?"))
print(sor(proc, model, IMG["c31"], "caption tr", 40))
print(f"GPU bellek: {torch.cuda.max_memory_allocated()/1e9:.1f} GB")

## 4. Ana döngü: her model için tüm deneyler

In [ ]:
os.makedirs("/content/sonuclar", exist_ok=True)
QA, OCR_R, CAP, FIS = [], [], [], []

for ad in MODELLER:
    kisa = ad.split("-")[-1]                      # "224" / "448"
    if "model" not in globals() or getattr(model, "_ad", None) != ad:
        model = proc = None; gc.collect(); torch.cuda.empty_cache()
        proc, model = yukle(ad); model._ad = ad
    print(f"\n=== {ad} ===")

    # Deney 1-2: tabela soru-cevap, TR ve EN
    for q in DATA["sorular"]:
        for dil in ["tr", "en"]:
            pred, sn = sor(proc, model, IMG[q["img"]], f"answer {dil} {q[dil]}")
            kati, esnek = puanla(pred, q["gold"], q["tip"])
            QA.append(dict(model=kisa, dil=dil, qid=q["qid"], img=q["img"], boy=q["boy"], tip=q["tip"],
                           soru=q[dil], gold=q["gold"], cevap=pred, kati=kati, esnek=esnek, sure=sn))
    print("tabela QA bitti")

    # Deney 3: OCR, Türkçe harfler
    for gid, gold in DATA["ocr"].items():
        pred, sn = sor(proc, model, IMG[gid], "ocr", 160)
        d, k, o = tr_harf(pred, gold)
        OCR_R.append(dict(model=kisa, img=gid, gold=gold, cevap=pred, cer=cer(pred, gold), cer_esnek=cer(pred, gold, True),
                          tr_dogru=d, tr_katlanmis=k, tr_diger=o, sure=sn))
    print("OCR bitti")

    # Nitel: altyazı
    for gid in DATA["altyazi"]:
        for dil in ["tr", "en"]:
            pred, sn = sor(proc, model, IMG[gid], f"caption {dil}", 64)
            CAP.append(dict(model=kisa, img=gid, dil=dil, cevap=pred, sure=sn))

    # Deney 4: fişler
    for f in cord:
        for dil, s in [("en", "What is the total price?"), ("tr", "Toplam tutar ne kadar?")]:
            pred, sn = sor(proc, model, f["img"], f"answer {dil} {s}")
            FIS.append(dict(model=kisa, dil=dil, fid=f["fid"], gold=f["gold"], cevap=pred, dogru=fis_dogru(pred, f["gold"]), sure=sn))
    print("fişler bitti")

    # her modelden sonra kaydet (bağlantı koparsa kaybolmasın)
    for adi, rows in [("qa", QA), ("ocr", OCR_R), ("altyazi", CAP), ("fis", FIS)]:
        pd.DataFrame(rows).to_csv(f"/content/sonuclar/{adi}.csv", index=False)
    print(f"GPU tepe bellek: {torch.cuda.max_memory_allocated()/1e9:.1f} GB")

## 5. Özet tablolar

In [ ]:
qa, ocr_df, cap, fis = (pd.DataFrame(x) for x in (QA, OCR_R, CAP, FIS))

print("### Deney 1-2: Tabela soru-cevap doğruluğu (%)  [katı / esnek]")
t = qa.groupby(["model", "dil"])[["kati", "esnek"]].mean().mul(100).round(1)
t["n"] = qa.groupby(["model", "dil"]).size(); display(t)

print("### Yazı boyuna göre (katı, %)")
display(qa.pivot_table(index="boy", columns=["model", "dil"], values="kati", aggfunc="mean").mul(100).round(1))

print("### Soru tipine göre (katı, %)")
display(qa.pivot_table(index="tip", columns=["model", "dil"], values="kati", aggfunc="mean").mul(100).round(1))

print("### Ortalama süre (sn/soru)")
display(qa.groupby("model")["sure"].mean().round(2))

In [ ]:
print("### Deney 3: OCR — karakter hata oranı (düşük = iyi)")
o = ocr_df.groupby("model")[["cer", "cer_esnek"]].mean().mul(100).round(1)
s = ocr_df.groupby("model")[["tr_dogru", "tr_katlanmis", "tr_diger"]].sum()
s["tr_dogru_%"] = (100 * s.tr_dogru / s.sum(axis=1)).round(1)
display(o.join(s))
display(ocr_df[["model", "img", "gold", "cevap", "cer"]])

In [ ]:
print("### Deney 4: Fişte toplam tutar doğruluğu (%)")
f = fis.groupby(["model", "dil"])["dogru"].mean().mul(100).round(1).to_frame()
f["n"] = fis.groupby(["model", "dil"]).size(); display(f)
display(fis.groupby("model")["sure"].mean().round(2))

In [ ]:
print("### 224 ile 448'in ayrıştığı sorular (TR soru)")
p = qa[qa.dil == "tr"].pivot_table(index=["qid", "img", "soru", "boy"], columns="model", values=["kati", "cevap"], aggfunc="first")
fark = p[p[("kati", "224")] != p[("kati", "448")]]
pd.set_option("display.max_colwidth", 60); display(fark)

In [ ]:
print("### Altyazılar (nitel)")
display(cap.pivot_table(index=["img", "dil"], columns="model", values="cevap", aggfunc="first"))

## 6. Yazı için örnek görseller

`goster("c44")` yazınca görsel ve iki modelin cevapları yan yana çıkıyor. Ekran görüntüsü alıp yazıda kullanabilirsin (caption'da görselin lisansını belirt, `DATA["gorseller"]["c44"]`).

In [ ]:
import matplotlib.pyplot as plt, textwrap
def goster(gid, dil="tr"):
    sat = qa[(qa.img == gid) & (qa.dil == dil)]
    fig, ax = plt.subplots(figsize=(7, 7)); ax.imshow(IMG[gid]); ax.axis("off")
    yazi = []
    for _, r in sat.groupby("qid").first().iterrows():
        yazi.append(f"S: {r.soru}   (doğru: {r.gold})")
        for m in ["224", "448"]:
            rr = sat[(sat.qid == r.name) & (sat.model == m)].iloc[0]
            yazi.append(f"   {m}: {rr.cevap}  {'✓' if rr.kati else '✗'}")
    g = DATA["gorseller"][gid]
    ax.set_title("\n".join(yazi), loc="left", fontsize=10, family="monospace")
    fig.text(0.01, 0.01, f"Fotoğraf: {g['yazar']}, {g['lisans']}, Wikimedia Commons", fontsize=7)
    plt.tight_layout(); plt.show()

for gid in fark.index.get_level_values("img").unique()[:6]:
    goster(gid)

In [ ]:
# Sonuçları indir (zip): bunu bana geri gönder, analizi ve yazı bölümünü birlikte yapalım
import shutil
from google.colab import files
shutil.make_archive("/content/paligemma2_sonuclar", "zip", "/content/sonuclar")
files.download("/content/paligemma2_sonuclar.zip")